# 🚗 Treinamento YOLO11m - Análise de Tráfego
**Google Colab - GPU Gratuita**

## Como usar:
1. Ative a GPU: **Runtime → Change runtime type → GPU → Save**
2. Execute cada célula em ordem
3. Baixe o modelo treinado ao final

## 📁 Fonte dos dados:
O dataset está incluído no repositório GitHub (`datasets/traffic_sample.zip`) com ~30 imagens de treino e 10 de validação.
Para treinar com suas próprias imagens, faça upload para o Google Drive e aponte o caminho no passo 2.

**Tempo estimado:** 30-60 minutos para 50 épocas com GPU.


In [ ]:
# ============================================================================
# PASSO 1: Instalar dependências
# ============================================================================
!pip install -q ultralytics opencv-python numpy
print("✅ Dependências instaladas")

In [ ]:
# ============================================================================
# PASSO 2: Baixar dataset do GitHub Release
# ============================================================================
import os

dataset_url = 'https://github.com/LanzaDev/traffic_analysis/releases/download/v1.0/dataset.zip'
print('📥 Baixando dataset do GitHub...')
!wget -q '{dataset_url}' -O dataset.zip

if os.path.exists('dataset.zip'):
    print(f'✅ Dataset baixado: {os.path.getsize("dataset.zip") / 1024 / 1024:.1f} MB')
    !unzip -q dataset.zip
    print('✅ Dataset extraído')
else:
    print('⚠️ Download falhou. Faça upload manual para o Google Drive e use o passo 2b.')


In [ ]:
# ============================================================================
# PASSO 2b (Opcional): Montar Google Drive para datasets maiores
# ============================================================================
from google.colab import drive

print('🔗 Conectando ao Google Drive...')
drive.mount('/content/drive')

print('\n📁 Estrutura do Drive:')
!ls -la /content/drive/MyDrive/ 2>/dev/null || echo 'Diretório vazio ou vazio'

print('\n💡 Coloque seus arquivos de treino em: /content/drive/MyDrive/traffic_dataset/')
print('   Estrutura esperada:')
    print('   /content/drive/MyDrive/traffic_dataset/')
    print('   ├── images/train/')
    print('   ├── images/val/')
    print('   └── labels/train/')
    print('   └── labels/val/')

In [ ]:
# ============================================================================
# PASSO 3: Criar arquivo data.yaml
# ============================================================================
import os

# Escolher fonte: dataset do GitHub (download.zip) ou Google Drive
use_drive = input('Usar Google Drive? (s/n): ').strip().lower() == 's'

if use_drive:
    data_path = '/content/drive/MyDrive/traffic_dataset'
else:
    data_path = 'dataset'

yaml_content = f'''train: {{data_path}}/images/train
val: {{data_path}}/images/val
nc: 6
names:
  0: person
  1: car
  2: motorcycle
  3: bus
  4: truck
  5: articulated_truck'''

with open('data.yaml', 'w') as f:
    f.write(yaml_content)

print('✅ Arquivo data.yaml criado:')
print(yaml_content)

# Verificar se os arquivos existem
train_dir = f'{data_path}/images/train'
val_dir = f'{data_path}/images/val'

if os.path.exists(train_dir):
    n_train = len([f for f in os.listdir(train_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
    print(f'📊 Treino: {n_train} imagens')
else:
    print(f'⚠️ Pasta não encontrada: {train_dir}')

if os.path.exists(val_dir):
    n_val = len([f for f in os.listdir(val_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
    print(f'📊 Validação: {n_val} imagens')
else:
    print(f'⚠️ Pasta não encontrada: {val_dir}')

In [ ]:
# ============================================================================
# PASSO 5: Treinar o modelo
# ============================================================================
import torch
from ultralytics import YOLO

device = '0' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('⚠️ AVISO: GPU não detectada. Treinando em CPU (será mais lento).')
else:
    print(f'✅ GPU detectada: {torch.cuda.get_device_name(0)}')

model = YOLO('yolo11m.pt')

results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project='runs/train',
    name='traffic_analysis',
    exist_ok=True,
    patience=20,
    save=True,
    verbose=True
)

print('\n✅ Treinamento concluído!')
print(f"📁 Modelo salvo em: runs/train/traffic_analysis/weights/best.pt")

In [ ]:
# ============================================================================
# PASSO 6: Validar o modelo treinado
# ============================================================================
from ultralytics import YOLO

model = YOLO('runs/train/traffic_analysis/weights/best.pt')

metrics = model.val(
    data='data.yaml',
    imgsz=640,
    batch=16,
    device=device
)

print('\n✅ Validação concluída!')
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precisão: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

In [ ]:
# ============================================================================
# PASSO 7: Baixar o modelo treinado
# ============================================================================
from google.colab import files

model_path = 'runs/train/traffic_analysis/weights/best.pt'

import os
if os.path.exists(model_path):
    print(f"📥 Baixando modelo: {model_path}")
    files.download(model_path)
    print('✅ Download iniciado!')
else:
    print('❌ Modelo não encontrado. Verifique se o treinamento foi concluído.')

# 📋 Como usar o modelo treinado no seu projeto

1. Baixe o arquivo `best.pt` que foi gerado acima
2. Substitua o arquivo `models/yolo11m.pt` no seu projeto pelo novo modelo
3. Execute normalmente:
   ```bat
   run.bat --mode vehicles --source assets/seu_video.mp4 --no-show
   ```

---
## 📚 Dicas para melhorar o resultado:
- Use pelo menos 100-200 imagens anotadas para treino
- Variedade de ângulos, iluminação e clima
- Anote apenas as classes que você precisa (car, truck, bus, motorcycle, person)
- Use `--conf 0.25` para detecção padrão ou `--conf 0.40` para detecção mais conservadora

---
🏁 **Fim do notebook**